# Step 4: Qualitative Evaluation
Compare BM25 and Semantic Search on the Amazon Appliances corpus.

## 1. Imports & path setup

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

sys.path.append(str(Path('..') / 'src'))

from bm25 import load_artefacts as load_bm25, bm25_search
from semantic import load_artefacts as load_semantic, semantic_search
from sentence_transformers import SentenceTransformer

/Users/wfrankel/miniforge3/envs/575_capstone/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load indexes from disk

In [2]:
# BM25
bm25_model, tokenized_corpus, chunks = load_bm25()

# Semantic
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
faiss_index, embeddings, _ = load_semantic()

print(f'Corpus size: {len(chunks):,} chunks')
chunks.head(2)

Loaded BM25 artefacts from disk.


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9740.78it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded FAISS index (1,149 vectors) from disk.
Corpus size: 1,149 chunks


,chunk_id,parent_asin,rating,chunk_text
0,0,B00F396J80,1.0,I was so excited to receive this product. I h...
1,1,B07CXWFMJH,5.0,I got this for my husband to use in his Franci...


## 3. Define query set

10 queries spanning easy (keyword), medium (semantic), and complex difficulty.

In [ ]:
queries = [
    # Easy: keyword-based 
    "temperature controlled fridge",
    "washing machine not draining water",
    "large microwave",

    # Medium: semantic 
    "appliance to cool down my home",
    "something to dry clothes without a dryer",
    "quiet espresso machine with grinder attached",

    # Complex: require reasoning / context 
    "my microwave is too loud!",
    "my child keeps tripping over my kettle wire and giving himself concussions",
    "what is the appliance where you put dishes in and it outputs clean dishes",
    "my grandma loves to bake, what should i get for her birthday?",
]

## 4. Run retrieval: top 5 results per query per method

In [5]:
def run_eval(queries, bm25_model, faiss_index, chunks, embed_model, top_k=5):
    results = []
    for q in queries:
        bm25_res = bm25_search(q, bm25_model, chunks, top_k=top_k)
        sem_res  = semantic_search(q, faiss_index, chunks, embed_model, top_k=top_k)
        results.append({
            "query": q,
            "bm25": bm25_res.to_dict(orient="records"),
            "semantic": sem_res.to_dict(orient="records"),
        })
    return results

results = run_eval(queries, bm25_model, faiss_index, chunks, embed_model)
print(f'Evaluated {len(results)} queries.')

Evaluated 10 queries.


## 5. Display results for all 10 queries

In [ ]:
# I asked claude to create a function to make a nice, printable output for all my queries, and this is the code it gave. 

def show_query(result):
    """Pretty-print one query result as a markdown table."""
    q = result['query']
    rows = []
    for rank, (b, s) in enumerate(zip(result['bm25'], result['semantic']), 1):
        b_text = b['chunk_text'][:120].replace('|', ' ').strip()
        s_text = s['chunk_text'][:120].replace('|', ' ').strip()
        rows.append(
            f"| {rank} "
            f"| {b_text}… (score={b['score']:.3f}, ★{b['rating']}) "
            f"| {s_text}… (score={s['score']:.3f}, ★{s['rating']}) |"
        )
    table = (
        f"### Query: *{q}*\n\n"
        "| Rank | BM25 | Semantic |\n"
        "|------|------|----------|\n"
        + "\n".join(rows)
    )
    display(Markdown(table))

for r in results:
    show_query(r)
    print()

### Query: *temperature controlled fridge*

| Rank | BM25 | Semantic |
|------|------|----------|
| 1 | I love my little refrigerator. I use it in my classroom. I love the can holder in the door too. It helps with storing al… (score=8.379, ★5.0) | This refrigerator is well designed. It is sleek and beautiful. Simple  to set up and install it maintains the temperatur… (score=0.613, ★5.0) |
| 2 | Does freeze everything very quickly. just after a 2 months of having it,  Ice started to build around the front. Was set… (score=7.505, ★3.0) | My husband keeps this refrigerator in his room at work. It helps keep his things more private, so it's less likely other… (score=0.589, ★5.0) |
| 3 | See attached photo. I zip locked 4 of these into a bag filled with Boveda 69% RH packs. After 24 hrs, ALL 4 are reading… (score=7.505, ★5.0) | Nice mini fridge. Drawers are nice in the refrigerator part and the door has nice holders for canned beverages. Freezer… (score=0.565, ★5.0) |
| 4 | This fridge is just a tad taller than our old mini fridge, and a very slight difference in width as well. With that said… (score=7.184, ★4.0) | This is the coolest refrigerator.  The double glass door allows me to see what inside with a simple glance.  It The door… (score=0.556, ★5.0) |
| 5 | Works, but some on screen print is beyond tiny! I regifted mine. Mengshen® Digital Temperature and Humidity Meter - with… (score=6.465, ★1.0) | Nice fridge that is small, but is great for a bunch of drinks and skin care. The light can be turned off also! The price… (score=0.552, ★5.0) |

### Query: *washing machine not draining water*

| Rank | BM25 | Semantic |
|------|------|----------|
| 1 | As a number of other reviews have noticed, there is a recurrent defect with some of these washers where the washer conti… (score=11.825, ★1.0) | As a number of other reviews have noticed, there is a recurrent defect with some of these washers where the washer conti… (score=0.611, ★1.0) |
| 2 | The Washing machine works great, the spin dry side spin's cloths dryer then any full sized I have ever used.  You have t… (score=11.360, ★4.0) | [[VIDEOID:38225654b62b7bb5f80d41c3eec3b6ef]] UPDATE: I've gotten the machine hooked up to the standard water & drain in… (score=0.532, ★3.0) |
| 3 | This machine is worth EVERY penny. My only regret so far, is not purchasing this sooner. Not only is it very lightweight… (score=10.505, ★5.0) | I have the washer less than 30 days. It already does NOT work. Used 3 times. Panda 5.5 lbs Counter Top Washing machine w… (score=0.502, ★1.0) |
| 4 | Would not fit the drain appendage on a very popular Frigidaire model I have.  A plumber confirmed I was replacing correc… (score=9.385, ★1.0) | The Washing machine works great, the spin dry side spin's cloths dryer then any full sized I have ever used.  You have t… (score=0.500, ★4.0) |
| 5 | This works really well, and it makes Sonic ice!  However, I do wish that I could attach a water line.  I have a regular… (score=8.190, ★3.0) | I am so grateful for the reviews! The tips and insights were helpful, so now this is a great product for me.<br />My was… (score=0.490, ★5.0) |

### Query: *large microwave*

| Rank | BM25 | Semantic |
|------|------|----------|
| 1 | Cooks very fast_<br />Potatoes: poke some holes in potato with a fork, place in tray, add a little water, top with lid,… (score=7.281, ★5.0) | I like the product everything was great a couple of dance but thank God we hide it with the cabinets they arrived late b… (score=0.462, ★5.0) |
| 2 | Perfect matchup with stainless steel appliances.  Easy wrap around with velcro.  Purchased for microwave as well.  Quick… (score=5.850, ★5.0) | Cooks very fast_<br />Potatoes: poke some holes in potato with a fork, place in tray, add a little water, top with lid,… (score=0.452, ★5.0) |
| 3 | If you live alone or with just one other person, this microwave accessory will be one of your favorite tools, especially… (score=5.365, ★5.0) | If you live alone or with just one other person, this microwave accessory will be one of your favorite tools, especially… (score=0.446, ★5.0) |
| 4 | There are three long wraps (stove, fridge) plus two shorter wraps (toaster and/or microwave), each with a strip of velcr… (score=5.365, ★3.0) | There are three long wraps (stove, fridge) plus two shorter wraps (toaster and/or microwave), each with a strip of velcr… (score=0.441, ★3.0) |
| 5 | I like the product everything was great a couple of dance but thank God we hide it with the cabinets they arrived late b… (score=5.175, ★5.0) | And this one is the one for me.  I'm shorter and the vertical style caused a bit of an issue for me to reach the back bu… (score=0.411, ★5.0) |

### Query: *appliance to cool down my home*

| Rank | BM25 | Semantic |
|------|------|----------|
| 1 | I was not having any luck finding a replacement pad for my Honeywell cool mist humidifier, and came across this. It look… (score=10.337, ★5.0) | This appliance is terrific. Fast and easy to use it produces bullet shaped ice. It fits easily on the counter and would… (score=0.511, ★5.0) |
| 2 | Mine is the chef trattoria one and as soon as I received it, I opened and unrolled the picture/magnet.  The directions s… (score=9.695, ★5.0) | Modern and cute looking, efficient mini fridge, which is a good size for me to hold drinks and my small food items. The… (score=0.510, ★5.0) |
| 3 | I vacillated between getting a four burner flat cooktop induction cooktop and a gas cooktop as I have always preferred g… (score=9.116, ★5.0) | Unit works great but ice melts fast so unit run almost all the time. I set this up on a Kasa smart switch and automated… (score=0.471, ★5.0) |
| 4 | This appliance is terrific. Fast and easy to use it produces bullet shaped ice. It fits easily on the counter and would… (score=8.929, ★5.0) | We just got this yesterday, and my daughter got a good night's rest up in her loft area. The area tends to get hotter th… (score=0.466, ★5.0) |
| 5 | [[VIDEOID:ba4541635d2545c9271a4d2adf78c0f9]] I love chewable Ice nuggets!  I will go to a certain restaurant just becaus… (score=8.388, ★5.0) | I like it.<br /><br />This has probably been the best user reviewed as well as economical model on the market today, but… (score=0.462, ★4.0) |

### Query: *something to dry clothes without a dryer*

| Rank | BM25 | Semantic |
|------|------|----------|
| 1 | Drying clothes is something we take for granted and we expect our clothes dryer to get our clothing dry the first time,… (score=18.994, ★4.0) | Works great for us, and saves a lot of space vs. having a separate washer + dryer.  Sometimes it doesn't get the clothes… (score=0.618, ★5.0) |
| 2 | First bunch of washes good, but lint starts to become a problem with clothes from prior washes. I cleaned the lint trap… (score=16.677, ★4.0) | this was bought as a gift. the person loves it. very convenient. Weekweed - Portable Folding Electric Air Drying Clothes… (score=0.608, ★5.0) |
| 3 | Works everytime. Good for small spaces. Uses regular electric outlet. It takes a longer time to dry than normal size dry… (score=16.240, ★5.0) | OMG I love this thing, its mesmerizing to watch it work haha.  I don't know why, I love doing my laundry now (with the P… (score=0.587, ★5.0) |
| 4 | This is great for small things. I use it to dry masks for my family. Small enough to always be set up in the corner...sm… (score=14.911, ★5.0) | this really works well, I use this for spinning out water from the clothes I wash in my portable washing machine. Also i… (score=0.587, ★5.0) |
| 5 | I've had my new dryer over a month now and it dries clothes great!! One single Haier 1.0 cu ft washer load is dry in Pan… (score=14.405, ★4.0) | The advertisement says that you can put gym clothes, swimsuit, vest and more in this.  Well this thing is not much bigge… (score=0.579, ★4.0) |

### Query: *quiet espresso machine with grinder attached*

| Rank | BM25 | Semantic |
|------|------|----------|
| 1 | Purchased this machine while renovating our tiny beach house.  I could fit no larger than this one.  I am totally deligh… (score=9.808, ★5.0) | Picked up the small Mr. Coffee espresso machine here from amazon and this pitcher is the perfect size to steam my milk.<… (score=0.466, ★5.0) |
| 2 | Works very well. Expected delivery date was July 20 to august 4 came very early on June 14 unbelievable. Washes very cle… (score=9.505, ★5.0) | I got this for my husband to use in his FrancisFrancis Y5 machine. This capsule fits perfectly and actually requires les… (score=0.447, ★5.0) |
| 3 | I got these as an experiment to replace my MyCaps brand filters for the Nespresso refills. They are slightly smaller, bu… (score=9.333, ★5.0) | I was so excited to use this for cold brew since I love so many OXO products, but this was a disappointment. I’ve been m… (score=0.438, ★1.0) |
| 4 | We've had K-cup coffee on and off for many years. We have certainly added our share of non-compostable waste to the land… (score=9.095, ★5.0) | I bought this elsewhere, but felt I had to at least post my experience. This is the LOUDEST washing machine I've ever he… (score=0.437, ★1.0) |
| 5 | I got this to put at the bottom of the Flair 58 portafilter. Using a shower screen at the top. It fits the top of the ba… (score=8.930, ★3.0) | I am very happy to have found these! My husband and I have an Esperta2 machine. I've been happy with the prefilled pods… (score=0.435, ★5.0) |

### Query: *my microwave is too loud!*

| Rank | BM25 | Semantic |
|------|------|----------|
| 1 | There are three long wraps (stove, fridge) plus two shorter wraps (toaster and/or microwave), each with a strip of velcr… (score=9.218, ★3.0) | If you live alone or with just one other person, this microwave accessory will be one of your favorite tools, especially… (score=0.384, ★5.0) |
| 2 | Cooks very fast_<br />Potatoes: poke some holes in potato with a fork, place in tray, add a little water, top with lid,… (score=7.281, ★5.0) | The ice maker on the refrigerator tends to act up and only make ice when it feels like it. This portable ice maker suits… (score=0.381, ★4.0) |
| 3 | The fit of these handle covers is average and also awkward  with regard to how the Velcro fits. I don't like the feel of… (score=6.609, ★3.0) | I bought the retro fridge and now purchased this to match.  Its not overly loud but it does make a small amount of noise… (score=0.375, ★5.0) |
| 4 | This VICKS HEALTHCHECK HUMIDITY AND TEMPERATURE MONITOR is a simple device that only does two things--display temperatur… (score=6.534, ★5.0) | Unable to use this washer for very long until it began making very loud noises.  I returned. COMFEE’ Washing Machine 2.4… (score=0.373, ★1.0) |
| 5 | Bought this a little over a year ago.  It worked fine, but then failed recently.  Decided to go OEM and then saw how thi… (score=6.452, ★2.0) | I bought this elsewhere, but felt I had to at least post my experience. This is the LOUDEST washing machine I've ever he… (score=0.362, ★1.0) |

### Query: *my child keeps tripping over my kettle wire and giving himself concussions*

| Rank | BM25 | Semantic |
|------|------|----------|
| 1 | OK, So my husband has a french press that has more parts to it.  He will use it during the weekend. Takes more time and… (score=13.370, ★5.0) | Starting to rust up on top. Returning !!! Milk Frothing Pitcher 20oz 600ml - Milk Jug 12 20 30oz - Measurements on Both… (score=0.237, ★1.0) |
| 2 | As a number of other reviews have noticed, there is a recurrent defect with some of these washers where the washer conti… (score=9.917, ★1.0) | A screw or some type of mechanism to keep the cap on would have been great. Finum Reusable Stainless Steel Coffee and Te… (score=0.236, ★5.0) |
| 3 | This is a good looking coffee drip system, and does work well.<br />My husband travels and said this type is a hassle ,… (score=9.319, ★3.0) | Not long after I put the proper amount of this into my water and ran my humidifier, my entire downstairs started smellin… (score=0.234, ★1.0) |
| 4 | GREAT FOR MY huMidifier!!! Keeps the water clear and the humidifier too!!! LOVE IT BestAir 3BT-PDQ-6 Original BT Humidif… (score=9.291, ★5.0) | I have two of these in my home.  It's been nothing but a constant stream of repairs and now the repair man refuses to wo… (score=0.227, ★1.0) |
| 5 | After not using my little washer for a few months I needed to use it while camping.  I am so irritated that it will not… (score=9.024, ★1.0) | The pitcher just arrived. It's packaged in a box, wrapped in a plastic bag. This would only be a problem if the carrier… (score=0.207, ★5.0) |

### Query: *what is the appliance where you put dishes in and it outputs clean dishes*

| Rank | BM25 | Semantic |
|------|------|----------|
| 1 | Before I purchased this, we had a magnet where we had to flip it over for clean and dirty. The magnet got ruined from ha… (score=24.056, ★5.0) | Looks sleek. Quiet. Dishes clean in about an hour. Fits on counters. Love having it!<br /><br />Just be sure to use soap… (score=0.547, ★5.0) |
| 2 | I’m using the magnet and it’s nice and strong; the thing doesn’t slip or slide around. The slide works smoothly. I like… (score=22.639, ★5.0) | I've been using this dishwasher for about 8 months and I absolutely love it!! I live alone so I need to run it 1-2 times… (score=0.519, ★5.0) |
| 3 | Bought this from Best Buy - Go there to read more reviews... this is not a loved product.<br /><br />The first one worke… (score=21.924, ★1.0) | This is a handy… Albeit kind of expensive… “Gadget”. Great for smaller places like studio apartments… Motorhomes… Dorm r… (score=0.513, ★5.0) |
| 4 | We actually intended to purchase the KUDE50FXSS because it is cheaper but our 9.5 yr old Fisher Paykel died at Thanksgiv… (score=21.628, ★4.0) | I've used portable dishwashers before and they are AWESOME for people who travel or have dietary restrictions (Kosher, f… (score=0.501, ★3.0) |
| 5 | I used this in an unorthadoxed way.<br /><br />I installed 2 dishwashers in my kitchen and needed a way to get the drain… (score=21.471, ★5.0) | This small dishwasher is ideal for small spaces. Set up was straightforward, not easy but not difficult either. It is re… (score=0.491, ★5.0) |

### Query: *my grandma loves to bake, what should i get for her birthday?*

| Rank | BM25 | Semantic |
|------|------|----------|
| 1 | My rating is based off the product itself. The box it came in was smashed up when it arrived. The packaging was not good… (score=17.668, ★4.0) | This was a gift and she loves it SOUKOO 2 in 1 Water Ice Maker Machine, 48lbs Daily Ice Cube Makers,Stainless Steel for… (score=0.351, ★5.0) |
| 2 | It's easy to use. We use bottled water. I thought that I'd be able to change the size of the ice with this one, I guess… (score=15.752, ★4.0) | My husband got tried of seeing old paper egg cartons in the refrigerator. He ordered these and I just love them. Very ni… (score=0.339, ★5.0) |
| 3 | This was bought as a Christmas present. Daughter loves to paint and draw and was keeping her brushes in a shoe box. This… (score=15.451, ★4.0) | These door handle covers are quite festive. It comes with a pair - Santa and Mrs. Clause. They don't fasten tight enough… (score=0.314, ★4.0) |
| 4 | I've had this for over a year now and have used it a lot.  It is pretty durable, I have it laying on the floor of my tru… (score=15.311, ★4.0) | I'm not sure why I waited so long to get one of these. I make bread and biscuits a lot and this sure makes clean up a br… (score=0.297, ★5.0) |
| 5 | I ordered this washer/dryer combo for my 85-year-old mom who lives 5 hours away from me.  After several years, the dryer… (score=15.007, ★5.0) | Great product for nursing care Hamilton Beach PIM-2-1A Portable Ice Maker, Candy Apple Red… (score=0.291, ★5.0) |